In [ ]:
import os
os.environ["HF_HOME"] = "/home/jovyan/.cache/huggingface"

import torch
import polars as pl
from pathlib import Path
import time
import numpy as np
from evo2 import Evo2

# configuration
DATA_ROOT = Path.home() / "vambersky_t/Data"
WINDOWS_DIR = DATA_ROOT / "extracted_windows"
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"


LAYER = "blocks.28.mlp.l3"
SEQ_LEN = 10_000
BIN_SIZE = 50
N_BINS = SEQ_LEN // BIN_SIZE
EMB_DIM = 4096

TFS_TO_PROCESS = Path("tfs_to_process.txt").read_text().splitlines()
CHECKPOINT_EVERY = 500

assert DATA_ROOT.exists(),   f"Data root missing: {DATA_ROOT}"
assert WINDOWS_DIR.exists(), f"Windows dir missing: {WINDOWS_DIR}"
assert N_BINS == 200

print(f"Bins: {N_BINS} × {BIN_SIZE} bp = {SEQ_LEN} bp")
print(f"Per-peak tensor: ({N_BINS}, {EMB_DIM}) float16")
print(f"Checkpoint interval: every {CHECKPOINT_EVERY} peaks")

# load model
print(f"Loading model...")
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model = Evo2("evo2_7b")
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [2]:
# get the embeddings - mean of every 50, non overlapping (probrat s Vojtou - rozlišení, porovnatelnost)

def embed_sequence(sequence: str) -> np.ndarray:
    """
    Forward-pass one 10 kb sequence through Evo 2, extract embeddings
    from LAYER, bin into (200, 4096) via reshape + mean, return as
    float16 numpy array.

    Uses module-level `model` and `LAYER`.
    """
    input_ids = torch.tensor(
        model.tokenizer.tokenize(sequence),
        dtype=torch.int,
    ).unsqueeze(0).to("cuda:0")

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[LAYER],
        )

    emb = embeddings[LAYER][0]                 # (10000, 4096)

    # reshape + mean - as i did in the test, this works.
    binned = emb.reshape(N_BINS, BIN_SIZE, -1).mean(dim=1)

    return binned.cpu().to(torch.float16).numpy()

# test, delete
test_df = pl.read_csv(
    next((WINDOWS_DIR / "CTCF").glob("*.full_peaks.csv.gz")),
    n_rows=1,
)
test_emb = embed_sequence(test_df["sequence"][0])
print(test_emb.shape, test_emb.dtype) 

(200, 4096) float16


In [3]:
# don't mess up my storage, pls

def get_output_paths(csv_path: Path) -> tuple[Path, Path, Path]:
    """
    Derive (final_npz, final_parquet, checkpoint_npz) from input CSV path.
    """
    base_name = csv_path.name.split(".full_peaks")[0]
    tf = base_name.split("__")[1]

    out_dir = EMBEDDINGS_DIR / tf
    out_dir.mkdir(parents=True, exist_ok=True)

    final_npz      = out_dir / f"{base_name}.embeddings.npz"
    final_parquet   = out_dir / f"{base_name}.peak_ids.parquet"
    checkpoint_npz  = out_dir / f"{base_name}.embeddings.checkpoint.npz"

    return final_npz, final_parquet, checkpoint_npz

# test, to be deleted
test_csv = next((WINDOWS_DIR / "CTCF").glob("*.full_peaks.csv.gz"))
for p in get_output_paths(test_csv):
    print(p)


/home/jovyan/vambersky_t/Data/embeddings/CTCF/ENCFF962IZR__CTCF__MCF-7.embeddings.npz
/home/jovyan/vambersky_t/Data/embeddings/CTCF/ENCFF962IZR__CTCF__MCF-7.peak_ids.parquet
/home/jovyan/vambersky_t/Data/embeddings/CTCF/ENCFF962IZR__CTCF__MCF-7.embeddings.checkpoint.npz


In [4]:
def process_csv_file(csv_path: Path) -> None:
    """
    Full embedding extraction for one CSV file of peak sequences.
    """
    final_npz, final_parquet, checkpoint_npz = get_output_paths(csv_path)

    if final_npz.exists() and final_parquet.exists(): # skip processed
        print(f"  SKIP (complete): {final_npz.name}")
        return
    
    df = pl.read_csv(csv_path)
    n_peaks = len(df)
    sequences = df["sequence"].to_list()
    peak_ids  = df["peak_id"].to_list()

    bad = [i for i, s in enumerate(sequences) if len(s) != SEQ_LEN] # check if length = 10 000 bp
    if bad:
        print(f"  WARNING: {len(bad)} sequences != {SEQ_LEN} bp "
              f"(indices {bad[:5]}{'...' if len(bad) > 5 else ''}). "
              f"These will be skipped.")

    start_idx = 0 # start from checkpoint? 
    if checkpoint_npz.exists():
        ckpt = np.load(checkpoint_npz, allow_pickle=True)
        collected      = list(ckpt["embeddings"])
        valid_peak_ids = list(ckpt["peak_ids"])
        start_idx      = int(ckpt["next_idx"])
        print(f"  RESUME from checkpoint: {start_idx}/{n_peaks} peaks processed, "
              f"{len(collected)} valid embeddings on disk")
    else:
        collected      = []
        valid_peak_ids = []

    
    # the LOOP!
    t0 = time.time()
    for i in range(start_idx, n_peaks):
        seq = sequences[i]

        if len(seq) != SEQ_LEN:
            continue

        emb = embed_sequence(seq)
        collected.append(emb)
        valid_peak_ids.append(peak_ids[i])

        if (i + 1) % 100 == 0:
            elapsed   = time.time() - t0
            done      = i + 1 - start_idx
            rate      = done / elapsed
            remaining = (n_peaks - i - 1) / rate if rate > 0 else float("inf")
            print(f"  [{i+1}/{n_peaks}] {rate:.1f} peaks/s  "
                  f"~{remaining / 60:.0f} min left  "
                  f"VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")


        if (i + 1) % CHECKPOINT_EVERY == 0:
            np.savez(
                checkpoint_npz,
                embeddings=np.stack(collected),
                peak_ids=np.array(valid_peak_ids, dtype=object),
                next_idx=np.array(i + 1),
            )
            print(f"  CHECKPOINT at peak {i+1}")


    # output
    embeddings_array = np.stack(collected)
    np.savez_compressed(final_npz, embeddings=embeddings_array)

    sidecar = (
        pl.DataFrame({"peak_id": valid_peak_ids})
        .join(
            df.select(["peak_id", "chr", "start", "end", "center"]),
            on="peak_id",
            how="left",
        )
    )
    sidecar.write_parquet(final_parquet)

    elapsed_total = time.time() - t0
    print(f"  DONE: {len(collected)} peaks → {final_npz.name}  "
          f"({embeddings_array.nbytes / 1e9:.2f} GB uncompressed)  "
          f"{elapsed_total / 60:.1f} min")

    if checkpoint_npz.exists():
        checkpoint_npz.unlink()
        print(f"  Checkpoint removed")




In [5]:
csv_files = []
for tf in TFS_TO_PROCESS:
    tf_dir = WINDOWS_DIR / tf
    if not tf_dir.exists():
        print(f"WARNING: No directory for {tf} at {tf_dir}")
        continue
    found = sorted(tf_dir.glob("*.full_peaks.csv.gz"))
    csv_files.extend(found)
    print(f"{tf}: {len(found)} file(s)")

print(f"\nTotal: {len(csv_files)} files")
for f in csv_files:
    print(f"  {f.parent.name}/{f.name}")

MYC: 8 file(s)
CTCF: 9 file(s)

Total: 17 files
  MYC/ENCFF043WTJ__MYC__K562.full_peaks.csv.gz
  MYC/ENCFF149BIS__MYC__MCF-7.full_peaks.csv.gz
  MYC/ENCFF356PWE__MYC__A549.full_peaks.csv.gz
  MYC/ENCFF372RQZ__MYC__H1.full_peaks.csv.gz
  MYC/ENCFF674RQX__MYC__A549.full_peaks.csv.gz
  MYC/ENCFF700CXD__MYC__H1.full_peaks.csv.gz
  MYC/ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz
  MYC/ENCFF800JFG__MYC__HepG2.full_peaks.csv.gz
  CTCF/ENCFF017XLW__CTCF__GM12878.full_peaks.csv.gz
  CTCF/ENCFF123DAC__CTCF__HepG2.full_peaks.csv.gz
  CTCF/ENCFF252PLM__CTCF__HepG2.full_peaks.csv.gz
  CTCF/ENCFF536RGD__CTCF__GM12878.full_peaks.csv.gz
  CTCF/ENCFF692RPA__CTCF__H1.full_peaks.csv.gz
  CTCF/ENCFF769AUF__CTCF__K562.full_peaks.csv.gz
  CTCF/ENCFF820BVW__CTCF__MCF-7.full_peaks.csv.gz
  CTCF/ENCFF962IZR__CTCF__MCF-7.full_peaks.csv.gz
  CTCF/ENCFF966KGI__CTCF__A549.full_peaks.csv.gz


In [6]:
test_csv = WINDOWS_DIR / "MYC" / "ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz"
print("START")
process_csv_file(test_csv)
print("END")

START
  [100/2216] 0.5 peaks/s  ~76 min left  VRAM 13.2 GB
  [200/2216] 0.5 peaks/s  ~72 min left  VRAM 13.2 GB
  [300/2216] 0.5 peaks/s  ~69 min left  VRAM 13.2 GB
  [400/2216] 0.5 peaks/s  ~65 min left  VRAM 13.2 GB
  [500/2216] 0.5 peaks/s  ~61 min left  VRAM 13.2 GB
  CHECKPOINT at peak 500
  [600/2216] 0.5 peaks/s  ~58 min left  VRAM 13.2 GB
  [700/2216] 0.5 peaks/s  ~54 min left  VRAM 13.2 GB
  [800/2216] 0.5 peaks/s  ~51 min left  VRAM 13.2 GB
  [900/2216] 0.5 peaks/s  ~47 min left  VRAM 13.2 GB
  [1000/2216] 0.5 peaks/s  ~44 min left  VRAM 13.2 GB
  CHECKPOINT at peak 1000
  [1100/2216] 0.5 peaks/s  ~40 min left  VRAM 13.2 GB
  [1200/2216] 0.5 peaks/s  ~37 min left  VRAM 13.2 GB
  [1300/2216] 0.5 peaks/s  ~33 min left  VRAM 13.2 GB
  [1400/2216] 0.5 peaks/s  ~29 min left  VRAM 13.2 GB
  [1500/2216] 0.5 peaks/s  ~26 min left  VRAM 13.2 GB
  CHECKPOINT at peak 1500
  [1600/2216] 0.5 peaks/s  ~22 min left  VRAM 13.2 GB
  [1700/2216] 0.5 peaks/s  ~19 min left  VRAM 13.2 GB
  [1800/

In [7]:
npz, parquet, _ = get_output_paths(test_csv)
data = np.load(npz)
ids = pl.read_parquet(parquet)
print(data["embeddings"].shape, data["embeddings"].dtype)
print(ids.shape)
print(ids.head())


(2216, 200, 4096) float16
(2216, 5)
shape: (5, 5)
┌─────────┬───────┬──────────┬──────────┬──────────┐
│ peak_id ┆ chr   ┆ start    ┆ end      ┆ center   │
│ ---     ┆ ---   ┆ ---      ┆ ---      ┆ ---      │
│ str     ┆ str   ┆ i64      ┆ i64      ┆ i64      │
╞═════════╪═══════╪══════════╪══════════╪══════════╡
│ peak_0  ┆ chr7  ┆ 4641937  ┆ 4642267  ┆ 4642102  │
│ peak_1  ┆ chr7  ┆ 44986420 ┆ 44986663 ┆ 44986541 │
│ peak_2  ┆ chr17 ┆ 17206300 ┆ 17206599 ┆ 17206449 │
│ peak_3  ┆ chr20 ┆ 38435139 ┆ 38435369 ┆ 38435254 │
│ peak_4  ┆ chr1  ┆ 92832060 ┆ 92832244 ┆ 92832152 │
└─────────┴───────┴──────────┴──────────┴──────────┘


In [ ]:
files_to_run = [
    WINDOWS_DIR / "MYC"  / "ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz",
    WINDOWS_DIR / "MYC"  / "ENCFF700CXD__MYC__H1.full_peaks.csv.gz",
    WINDOWS_DIR / "CTCF" / "ENCFF017XLW__CTCF__GM12878.full_peaks.csv.gz",
]

for i, csv_path in enumerate(files_to_run):
    print(f"\n[{i+1}/{len(files_to_run)}] {csv_path.parent.name}/{csv_path.name}")
    process_csv_file(csv_path)
    torch.cuda.empty_cache()


[1/3] MYC/ENCFF765CKK__MYC__GM12878.full_peaks.csv.gz
  SKIP (complete): ENCFF765CKK__MYC__GM12878.embeddings.npz

[2/3] MYC/ENCFF700CXD__MYC__H1.full_peaks.csv.gz
  [100/4957] 0.5 peaks/s  ~174 min left  VRAM 13.2 GB
  [200/4957] 0.5 peaks/s  ~171 min left  VRAM 13.2 GB


In [ ]:

# for idx, csv_path in enumerate(csv_files):
#     print(f"\n[{idx+1}/{len(csv_files)}] {csv_path.parent.name}/{csv_path.name}")
#     process_csv_file(csv_path)
#     torch.cuda.empty_cache()

# elapsed = time.time() - pipeline_t0
# print(f"\n{'='*50}")
# print(f"Pipeline complete: {elapsed / 3600:.1f} hours")
